In [ ]:
import sys
from google.colab import drive
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/MCX_data'
sys.path.append(folder_path)
import pandas as pd
import glob
import os
import numpy as np
import sys
import pickle
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from sklearn.preprocessing import StandardScaler

Mounted at /content/drive


### Read the CSV

In [ ]:
import sys
! pip install pmcx
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pickle
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
RANDOM_STATE = 42
MAPE_EPS = 1e-8
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 2.5 MB/s eta 0:00:00


In [ ]:
import os
csv_path = "/content/drive/MyDrive/MCX_data/result_folder/training_fd_110MHz.csv"
df = pd.read_csv(csv_path)
distances = [10, 20, 30, 40]
metrics = ["uac", "udc", "phase_rad"]
sub = df[df["sds_key"].isin(distances)].copy()

wl_order = sorted(sub["wavelength_index"].unique())
ids = np.array(sorted(sub["simulation_id"].unique()))

feature_blocks = []
feature_names = []

for d in distances:
    for wl_idx in wl_order:
        tmp = (
            sub[(sub["sds_key"] == d) & (sub["wavelength_index"] == wl_idx)]
            .set_index("simulation_id")
            .loc[ids, metrics]
        )

        feature_blocks.append(tmp.to_numpy(dtype=np.float64))

        wl_label = f"wl{wl_idx + 1}"
        feature_names.extend([
            f"uac_d{d}_{wl_label}",
            f"udc_d{d}_{wl_label}",
            f"phase_d{d}_{wl_label}"
        ])

X = np.concatenate(feature_blocks, axis=1)

print(X.shape)
print(feature_names)

np.save("fd_features_Nx24.npy", X)
np.save("fd_features_ids.npy", ids)

out_df = pd.DataFrame(X, columns=feature_names)
training_set = out_df

(10000, 24)
['uac_d10_wl1', 'udc_d10_wl1', 'phase_d10_wl1', 'uac_d10_wl2', 'udc_d10_wl2', 'phase_d10_wl2', 'uac_d20_wl1', 'udc_d20_wl1', 'phase_d20_wl1', 'uac_d20_wl2', 'udc_d20_wl2', 'phase_d20_wl2', 'uac_d30_wl1', 'udc_d30_wl1', 'phase_d30_wl1', 'uac_d30_wl2', 'udc_d30_wl2', 'phase_d30_wl2', 'uac_d40_wl1', 'udc_d40_wl1', 'phase_d40_wl1', 'uac_d40_wl2', 'udc_d40_wl2', 'phase_d40_wl2']


In [ ]:
csv_path = "/content/drive/MyDrive/MCX_data/result_folder/testing_fd_110MHz.csv"
df = pd.read_csv(csv_path)
distances = [10, 20, 30, 40]
metrics = ["uac", "udc", "phase_rad"]
sub = df[df["sds_key"].isin(distances)].copy()
wl_order = sorted(sub["wavelength_index"].unique())
ids = np.array(sorted(sub["simulation_id"].unique()))

feature_blocks = []
feature_names = []

for d in distances:
    for wl_idx in wl_order:
        tmp = (
            sub[(sub["sds_key"] == d) & (sub["wavelength_index"] == wl_idx)]
            .set_index("simulation_id")
            .loc[ids, metrics]
        )

        feature_blocks.append(tmp.to_numpy(dtype=np.float64))

        wl_label = f"wl{wl_idx + 1}"
        feature_names.extend([
            f"uac_d{d}_{wl_label}",
            f"udc_d{d}_{wl_label}",
            f"phase_d{d}_{wl_label}"
        ])

X = np.concatenate(feature_blocks, axis=1)

print(X.shape)
print(feature_names)

np.save("fd_features_Nx24.npy", X)
np.save("fd_features_ids.npy", ids)

out_df = pd.DataFrame(X, columns=feature_names)
#out_df.insert(0, "simulation_id", ids)
#out_df.to_csv("fd_features_Nx24_with_ids_testing.csv", index=False)
testing_set = out_df

(1000, 24)
['uac_d10_wl1', 'udc_d10_wl1', 'phase_d10_wl1', 'uac_d10_wl2', 'udc_d10_wl2', 'phase_d10_wl2', 'uac_d20_wl1', 'udc_d20_wl1', 'phase_d20_wl1', 'uac_d20_wl2', 'udc_d20_wl2', 'phase_d20_wl2', 'uac_d30_wl1', 'udc_d30_wl1', 'phase_d30_wl1', 'uac_d30_wl2', 'udc_d30_wl2', 'phase_d30_wl2', 'uac_d40_wl1', 'udc_d40_wl1', 'phase_d40_wl1', 'uac_d40_wl2', 'udc_d40_wl2', 'phase_d40_wl2']


### GT

In [ ]:
GT_folder_train = '/content/drive/MyDrive/MCX_data/stage2_csv/'
GT_folder_test = '/content/drive/MyDrive/MCX_data/test_csv/'

In [ ]:
csv_files_train = glob.glob(os.path.join(GT_folder_train, '*.csv'))
GT_all_train = pd.concat([pd.read_csv(f) for f in csv_files_train], ignore_index=True)
csv_files_test = glob.glob(os.path.join(GT_folder_test, '*.csv'))
GT_all_test = pd.concat([pd.read_csv(f) for f in csv_files_test], ignore_index=True)

In [ ]:
sorted_ids = [i + 1 for i in range(10000)]

# ensure ID is int
GT_all_train["ID"] = GT_all_train["ID"].astype(int)

# filter + order by ID = 1..10000
GT_filtered = (GT_all_train[GT_all_train["ID"].isin(sorted_ids)]
               .copy()
               .set_index("ID")
               .loc[sorted_ids]
               .reset_index())

target_cols = ["HBO1","HHB1", "HBO2", "HHB2", "d1", "a1", "a2", "b1", "b2"]
# (optional) verify all columns exist
missing = [c for c in target_cols if c not in GT_filtered.columns]
if missing:
    raise KeyError(f"Missing columns in GT_filtered: {missing}. Available: {list(GT_filtered.columns)}")

Y = GT_filtered[target_cols].to_numpy(dtype=np.float32)  # shape (N, 5)
Y_train = Y  # keep as (N,5) for multi-output regression

print("y_train shape:", Y_train.shape)
print("first row:", dict(zip(target_cols, Y_train[0])))

y_train shape: (10000, 9)
first row: {'HBO1': np.float32(10.618102), 'HHB1': np.float32(12.007143), 'HBO2': np.float32(46.95982), 'HHB2': np.float32(26.97317), 'd1': np.float32(12.0), 'a1': np.float32(1.8359671), 'a2': np.float32(1.2800592), 'b1': np.float32(2.1788228), 'b2': np.float32(2.103345)}


In [ ]:
sorted_ids = [i + 1 for i in range(1000)]
# ensure ID is int
GT_all_test["ID"] = GT_all_test["ID"].astype(int)

# filter + order by ID = 1..10000
GT_filtered = (GT_all_test[GT_all_test["ID"].isin(sorted_ids)]
               .copy()
               .set_index("ID")
               .loc[sorted_ids]
               .reset_index())

target_cols = ["HBO1","HHB1", "HBO2", "HHB2", "d1", "a1", "a2", "b1", "b2"]   # <-- adjust if needed

# (optional) verify all columns exist
missing = [c for c in target_cols if c not in GT_filtered.columns]
if missing:
    raise KeyError(f"Missing columns in GT_filtered: {missing}. Available: {list(GT_filtered.columns)}")

Y = GT_filtered[target_cols].to_numpy(dtype=np.float32)  # shape (N, 5)
Y_test = Y  # keep as (N,5) for multi-output regression

print("y_test shape:", Y_test.shape)
print("first row:", dict(zip(target_cols, Y_test[0])))

y_test shape: (1000, 9)
first row: {'HBO1': np.float32(10.618102), 'HHB1': np.float32(12.007143), 'HBO2': np.float32(46.95982), 'HHB2': np.float32(26.97317), 'd1': np.float32(12.0), 'a1': np.float32(1.8359671), 'a2': np.float32(1.2800592), 'b1': np.float32(2.1788228), 'b2': np.float32(2.103345)}


In [ ]:
X_train = training_set
X_test = testing_set

In [ ]:
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)

(10000, 24) (1000, 24) (10000, 9) (1000, 9)


### Gaussian noise adding

In [ ]:
def add_fdnirs_noise(X, acdc_rel, phase_std_rad, rng, clip_nonnegative=True):
    X = np.asarray(X, dtype=np.float64).copy()

    ac_idx = np.arange(0, X.shape[1], 3)
    dc_idx = np.arange(1, X.shape[1], 3)
    phase_idx = np.arange(2, X.shape[1], 3)

    # AC noise: relative Gaussian
    ac = X[:, ac_idx]
    ac_noise = rng.normal(loc=0.0, scale=np.abs(acdc_rel * ac), size=ac.shape)
    X[:, ac_idx] = ac + ac_noise

    # DC noise: relative Gaussian
    dc = X[:, dc_idx]
    dc_noise = rng.normal(loc=0.0, scale=np.abs(acdc_rel * dc), size=dc.shape)
    X[:, dc_idx] = dc + dc_noise

    # Phase noise: absolute Gaussian, in radians
    phase = X[:, phase_idx]
    phase_noise = rng.normal(loc=0.0, scale=phase_std_rad, size=phase.shape)
    X[:, phase_idx] = phase + phase_noise

    if clip_nonnegative:
        X[:, ac_idx] = np.clip(X[:, ac_idx], 0, None)
        X[:, dc_idx] = np.clip(X[:, dc_idx], 0, None)

    return X


# ============================================================
# Four noise levels
# AC/DC: relative noise
# Phase: literature values given in degrees -> convert to radians
# ============================================================

acdc_rel_levels = [0.0075, 0.015, 0.0375, 0.075]
phase_std_deg_levels = [0.15, 0.3, 0.75, 1.5]
phase_std_rad_levels = np.deg2rad(phase_std_deg_levels)  # radians

print("Phase std in degrees :", phase_std_deg_levels)
print("Phase std in radians :", phase_std_rad_levels)


# Use different RNGs so train/test noise are independent
rng_train = np.random.default_rng(123)
rng_test = np.random.default_rng(42)

# ============================================================
# Build noisy train/test sets
# ============================================================

noisy_trainsets = {}
noisy_testsets = {}

for level, (r, phase_std_rad) in enumerate(zip(acdc_rel_levels, phase_std_rad_levels), start=1):
    noisy_trainsets[f"level_{level}"] = add_fdnirs_noise(
        X=X_train,
        acdc_rel=r,
        phase_std_rad=phase_std_rad,
        rng=rng_train,
        clip_nonnegative=True,
    )

    noisy_testsets[f"level_{level}"] = add_fdnirs_noise(
        X=X_test,
        acdc_rel=r,
        phase_std_rad=phase_std_rad,
        rng=rng_test,
        clip_nonnegative=True,
    )

# ============================================================
# Unpack
# ============================================================

X_train_noise_1 = noisy_trainsets["level_1"]
X_train_noise_2 = noisy_trainsets["level_2"]
X_train_noise_3 = noisy_trainsets["level_3"]
X_train_noise_4 = noisy_trainsets["level_4"]

X_test_noise_1 = noisy_testsets["level_1"]
X_test_noise_2 = noisy_testsets["level_2"]
X_test_noise_3 = noisy_testsets["level_3"]
X_test_noise_4 = noisy_testsets["level_4"]

print("Train shape original :", X_train.shape)
print("Train noise 1 shape  :", X_train_noise_1.shape)
print("Train noise 2 shape  :", X_train_noise_2.shape)
print("Train noise 3 shape  :", X_train_noise_3.shape)
print("Train noise 4 shape  :", X_train_noise_4.shape)

print("Test shape original  :", X_test.shape)
print("Test noise 1 shape   :", X_test_noise_1.shape)
print("Test noise 2 shape   :", X_test_noise_2.shape)
print("Test noise 3 shape   :", X_test_noise_3.shape)
print("Test noise 4 shape   :", X_test_noise_4.shape)

Phase std in degrees : [0.15, 0.3, 0.75, 1.5]
Phase std in radians : [0.00261799 0.00523599 0.01308997 0.02617994]
Train shape original : (10000, 24)
Train noise 1 shape  : (10000, 24)
Train noise 2 shape  : (10000, 24)
Train noise 3 shape  : (10000, 24)
Train noise 4 shape  : (10000, 24)
Test shape original  : (1000, 24)
Test noise 1 shape   : (1000, 24)
Test noise 2 shape   : (1000, 24)
Test noise 3 shape   : (1000, 24)
Test noise 4 shape   : (1000, 24)


### Boost methods

In [ ]:
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
! pip install catboost
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
MAPE_EPS = 1e-8

RANDOM_STATE = 42
MAPE_EPS = 1e-8

HAS_XGB = True
HAS_LGBM = True
HAS_CAT = True

try:
    from xgboost import XGBRegressor
except Exception as e:
    HAS_XGB = False
    print(f"[skip] xgboost not available: {e}")

try:
    from catboost import CatBoostRegressor
except Exception as e:
    HAS_CAT = False
    print(f"[skip] catboost not available: {e}")

RANDOM_STATE = 42

# -------------------------
# Helpers
# -------------------------
def _wrap_multioutput_if_needed(base_estimator, Y):
    """
    Wrap estimator for multi-output regression when needed.
    XGBoost / LightGBM need this for multi-target regression.
    """
    Y = np.asarray(Y)
    if Y.ndim == 1 or (Y.ndim == 2 and Y.shape[1] == 1):
        return base_estimator
    return MultiOutputRegressor(base_estimator, n_jobs=-1)


## Other methods-No normalization

In [ ]:
def build_fixed_models():
    models = {}

    # Random Forest: supports multi-output directly
    models["random_forest_tuned"] = RandomForestRegressor(
        n_estimators=1000,
        max_depth=20,
        min_samples_split=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    # XGBoost
    if HAS_XGB:
        xgb_base = XGBRegressor(
            objective="reg:squarederror",
            booster="gbtree",
            tree_method="hist",
            device="cuda",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0,
            learning_rate=0.03,
            max_depth=4,
            n_estimators=1000,
        )
        models["xgboost_tuned"] = MultiOutputRegressor(xgb_base)

    GB_base = GradientBoostingRegressor(
    learning_rate=0.1,
    n_estimators=1000,
    max_depth=2,
    random_state=RANDOM_STATE)
    models["GB_base_tuned"] = MultiOutputRegressor(GB_base)

    models["catboost_cpu_tuned"] = CatBoostRegressor(
        loss_function="MultiRMSE",
        eval_metric="MultiRMSE",
        task_type="CPU",
        thread_count=-1,
        random_seed=RANDOM_STATE,
        verbose=False,
        learning_rate=0.1,
        depth=4,
        iterations=1000,
    )

    return models


# ============================================================
# Error calculations
# ============================================================

def evaluate_selected_metrics(Y_true, Y_pred, target_numbers_1based=(3, 4), mape_eps=1e-8):
    """
    target_numbers_1based=(3,4) matches your latest printed output.
    Change to (2,3) if you want the 2nd and 3rd targets instead.
    """
    Y_true = np.asarray(Y_true, dtype=np.float64)
    Y_pred = np.asarray(Y_pred, dtype=np.float64)

    if Y_true.ndim == 1:
        Y_true = Y_true.reshape(-1, 1)
    if Y_pred.ndim == 1:
        Y_pred = Y_pred.reshape(-1, 1)

    if Y_true.shape != Y_pred.shape:
        raise ValueError(f"Shape mismatch: Y_true {Y_true.shape}, Y_pred {Y_pred.shape}")

    target_idx = [t - 1 for t in target_numbers_1based]
    for idx in target_idx:
        if idx < 0 or idx >= Y_true.shape[1]:
            raise ValueError(f"Requested target index {idx} is out of range for Y with shape {Y_true.shape}")
    err = Y_pred - Y_true
    abs_err = np.abs(err)
    denom = np.maximum(np.abs(Y_true), mape_eps)
    ape = (abs_err / denom) * 100.0
    rows = []
    # Overall = per-sample mean across all targets
    overall_mae_per_sample = abs_err.mean(axis=1)
    overall_mape_per_sample = ape.mean(axis=1)

    rows.append({
        "item": "overall",
        "MAE_mean": float(overall_mae_per_sample.mean()),
        "MAE_std": float(overall_mae_per_sample.std()),
        "MAPE_mean": float(overall_mape_per_sample.mean()),
        "MAPE_std": float(overall_mape_per_sample.std()),
    })

    # Selected targets
    for tnum, idx in zip(target_numbers_1based, target_idx):
        rows.append({
            "item": f"target_{tnum}",
            "MAE_mean": float(abs_err[:, idx].mean()),
            "MAE_std": float(abs_err[:, idx].std()),
            "MAPE_mean": float(ape[:, idx].mean()),
            "MAPE_std": float(ape[:, idx].std()),
        })

    return rows


def print_metric_rows(header_name, rows):
    print(f"\n===== {header_name} =====")
    for r in rows:
        print(
            f"{r['item']:>10s} | "
            f"MAE = {r['MAE_mean']:.6f} ± {r['MAE_std']:.6f} | "
            f"MAPE = {r['MAPE_mean']:.6f} ± {r['MAPE_std']:.6f}"
        )


# ============================================================
# 4) Fit/evaluate one model on one matched noisy pair
# ============================================================
def fit_and_predict_one_model(model_name, model, X_train, Y_train, X_test):
    # norm_stats = fit_typewise_normalizer(X_train)
    # X_train_norm = transform_typewise(X_train, norm_stats)
    # X_test_norm = transform_typewise(X_test, norm_stats)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)
    return Y_pred


# ============================================================
# 5) Run on four matched noisy train/test pairs
# ============================================================

def run_fixed_models_on_noise_levels(
    Y_train,
    Y_test,
    train_test_pairs,
    target_numbers_1based=(3, 4),
    mape_eps=1e-8,
):
    models = build_fixed_models()

    if len(models) == 0:
        raise RuntimeError("No model is available. Check that xgboost is installed if you want to run XGBoost.")

    all_rows = []

    for noise_level, (X_train_cur, X_test_cur) in train_test_pairs.items():
        for model_name, model in models.items():
            Y_pred_cur = fit_and_predict_one_model(
                model_name=model_name,
                model=model,
                X_train=X_train_cur,
                Y_train=Y_train,
                X_test=X_test_cur,
            )

            rows = evaluate_selected_metrics(
                Y_true=Y_test,
                Y_pred=Y_pred_cur,
                target_numbers_1based=target_numbers_1based,
                mape_eps=mape_eps,
            )

            print_metric_rows(f"{noise_level} | {model_name}", rows)

            for r in rows:
                all_rows.append({
                    "noise_level": noise_level,
                    "model": model_name,
                    **r
                })
    results_df = pd.DataFrame(all_rows)
    return results_df

In [ ]:
# ============================================================
# 6) Matched noisy train/test pairs
#    Assumes these already exist:
#    X_train_noise_1 ... X_train_noise_4
#    X_test_noise_1  ... X_test_noise_4
# ============================================================

train_test_pairs = {
    "noise_level_1": (X_train_noise_1, X_test_noise_1),
    "noise_level_2": (X_train_noise_2, X_test_noise_2),
    "noise_level_3": (X_train_noise_3, X_test_noise_3),
    "noise_level_4": (X_train_noise_4, X_test_noise_4),
}


# ============================================================
# 7) Run
# ============================================================

results_df = run_fixed_models_on_noise_levels(
    Y_train=Y_train,
    Y_test=Y_test,
    train_test_pairs=train_test_pairs,
    target_numbers_1based=(3, 4),
    mape_eps=MAPE_EPS,
)

print("\nSummary table:")
print(results_df)

summary_pivot = results_df.pivot_table(
    index=["noise_level", "model"],
    columns="item",
    values=["MAE_mean", "MAE_std", "MAPE_mean", "MAPE_std"]
)

print("\nPivoted summary:")
print(summary_pivot)

overall_only = results_df[results_df["item"] == "overall"].copy()
overall_only = overall_only.sort_values(["noise_level", "MAE_mean", "MAPE_mean"])

print("\nSorted overall summary:")
print(overall_only[["noise_level", "model", "MAE_mean", "MAE_std", "MAPE_mean", "MAPE_std"]])


===== noise_level_1 | random_forest_tuned =====
   overall | MAE = 1.540066 ± 0.531821 | MAPE = 21.233373 ± 12.401302
  target_3 | MAE = 5.313532 ± 3.447056 | MAPE = 14.298619 ± 10.824865
  target_4 | MAE = 3.764142 ± 2.311865 | MAPE = 16.395087 ± 12.295147

===== noise_level_1 | xgboost_tuned =====
   overall | MAE = 1.726844 ± 0.592705 | MAPE = 20.637451 ± 11.093189
  target_3 | MAE = 6.596252 ± 4.095333 | MAPE = 17.715543 ± 12.977411
  target_4 | MAE = 4.626273 ± 2.748130 | MAPE = 20.132506 ± 14.716450

===== noise_level_1 | GB_base_tuned =====
   overall | MAE = 1.741953 ± 0.593566 | MAPE = 20.533721 ± 10.950477
  target_3 | MAE = 6.699584 ± 4.103987 | MAPE = 17.977163 ± 12.983380
  target_4 | MAE = 4.685347 ± 2.791654 | MAPE = 20.435900 ± 15.118019

===== noise_level_1 | catboost_cpu_tuned =====
   overall | MAE = 1.821324 ± 0.592076 | MAPE = 23.470199 ± 12.927760
  target_3 | MAE = 6.640091 ± 4.070660 | MAPE = 17.867453 ± 12.990190
  target_4 | MAE = 4.701525 ± 2.724185 | MAPE =

In [ ]:
summary_pivot

MAE_mean                       MAE_std  \
item                                overall  target_3  target_4   overall   
noise_level   model                                                         
noise_level_1 GB_base_tuned        1.741953  6.699584  4.685347  0.593566   
              catboost_cpu_tuned   1.821324  6.640091  4.701525  0.592076   
              random_forest_tuned  1.540066  5.313532  3.764142  0.531821   
              xgboost_tuned        1.726844  6.596252  4.626273  0.592705   
noise_level_2 GB_base_tuned        1.806226  6.883728  4.758199  0.601319   
              catboost_cpu_tuned   1.866821  6.803980  4.757562  0.605431   
              random_forest_tuned  1.703858  5.806903  4.104320  0.570745   
              xgboost_tuned        1.782141  6.739602  4.638130  0.607571   
noise_level_3 GB_base_tuned        1.934152  7.115421  4.857784  0.630876   
              catboost_cpu_tuned   1.970312  6.973243  4.795893  0.633749   
              random_forest_tuned  1.958204  6.535030  4.545387  0.621060   
              xgboost_tuned        1.924161  7.031293  4.805356  0.628322   
noise_level_4 GB_base_tuned        2.042129  7.176428  4.950136  0.640373   
              catboost_cpu_tuned   2.075107  7.119801  4.905607  0.638927   
              random_forest_tuned  2.136648  6.918034  4.812959  0.643704   
              xgboost_tuned        2.045115  7.169829  4.919595  0.641898   

                                                       MAPE_mean             \
item                               target_3  target_4    overall   target_3   
noise_level   model                                                           
noise_level_1 GB_base_tuned        4.103987  2.791654  20.533721  17.977163   
              catboost_cpu_tuned   4.070660  2.724185  23.470199  17.867453   
              random_forest_tuned  3.447056  2.311865  21.233373  14.298619   
              xgboost_tuned        4.095333  2.748130  20.637451  17.715543   
noise_level_2 GB_base_tuned        4.167718  2.830371  21.496825  18.474929   
              catboost_cpu_tuned   4.122555  2.763613  23.805344  18.285378   
              random_forest_tuned  3.705174  2.484825  23.508779  15.625891   
              xgboost_tuned        4.159902  2.751832  21.573686  18.091547   
noise_level_3 GB_base_tuned        4.265756  2.860860  23.751764  19.082128   
              catboost_cpu_tuned   4.222672  2.821668  25.575258  18.746759   
              random_forest_tuned  4.008176  2.694361  27.209106  17.503801   
              xgboost_tuned        4.219369  2.851001  24.065341  18.861708   
noise_level_4 GB_base_tuned        4.292910  2.916627  25.876614  19.314367   
              catboost_cpu_tuned   4.275021  2.875711  27.211473  19.223153   
              random_forest_tuned  4.243090  2.800906  29.925172  18.658539   
              xgboost_tuned        4.312883  2.910125  26.227756  19.356861   

                                               MAPE_std                        
item                                target_4    overall   target_3   target_4  
noise_level   model                                                            
noise_level_1 GB_base_tuned        20.435900  10.950477  12.983380  15.118019  
              catboost_cpu_tuned   20.469193  12.927760  12.990190  14.748267  
              random_forest_tuned  16.395087  12.401302  10.824865  12.295147  
              xgboost_tuned        20.132506  11.093189  12.977411  14.716450  
noise_level_2 GB_base_tuned        20.745524  11.432622  13.304625  15.356026  
              catboost_cpu_tuned   20.738760  13.002935  13.170043  15.006845  
              random_forest_tuned  17.859635  13.474153  11.725338  13.226375  
              xgboost_tuned        20.244999  11.401005  13.205289  14.899982  
noise_level_3 GB_base_tuned        21.200013  12.664520  13.600490  15.557024  
              catboost_cpu_tuned   20.904950  14.250170  13.475446  15.300027  
              random_forest_t

In [ ]:
overall_only

,noise_level,model,item,MAE_mean,MAE_std,MAPE_mean,MAPE_std
0,noise_level_1,random_forest_tuned,overall,1.540066,0.531821,21.233373,12.401302
3,noise_level_1,xgboost_tuned,overall,1.726844,0.592705,20.637451,11.093189
6,noise_level_1,GB_base_tuned,overall,1.741953,0.593566,20.533721,10.950477
9,noise_level_1,catboost_cpu_tuned,overall,1.821324,0.592076,23.470199,12.927760
12,noise_level_2,random_forest_tuned,overall,1.703858,0.570745,23.508779,13.474153
15,noise_level_2,xgboost_tuned,overall,1.782141,0.607571,21.573686,11.401005
18,noise_level_2,GB_base_tuned,overall,1.806226,0.601319,21.496825,11.432622
21,noise_level_2,catboost_cpu_tuned,overall,1.866821,0.605431,23.805344,13.002935
27,noise_level_3,xgboost_tuned,overall,1.924161,0.628322,24.065341,13.046223
30,noise_level_3,GB_base_tuned,overall,1.934152,0.630876,23.751764,12.664520


In [ ]:
# Save the pivot table directly
summary_pivot.to_csv(
    "/content/drive/MyDrive/summary_pivot.csv"
)

print("Saved to: /content/drive/MyDrive/summary_pivot.csv")

Saved to: /content/drive/MyDrive/summary_pivot.csv
